# 04 Popularity (baseline)

In [1]:
import sys
import time
import numpy as np
import pandas as pd
sys.path.append("..")          # cf_common.py is in the folder above

from cf_common import load_data, top_n, count_hits, precision_recall, save_results, BATCH_SIZE, TOP_KS

data = load_data()

476,451 users, 120,650 items, 10,000 users are evaluated


## Recommend the most popular tracks

In [2]:
n_recs = max(TOP_KS)
recs = np.zeros((len(data.eval_users), n_recs), dtype=np.int64)

start = time.perf_counter()
for first in range(0, len(data.eval_users), BATCH_SIZE):
    users = data.eval_users[first:first + BATCH_SIZE]
    scores = np.tile(data.popularity, (len(users), 1))       # the same scores for every user
    scores[data.ratings[users].nonzero()] = -np.inf          # remove what the user already has
    recs[first:first + len(users)] = top_n(scores, n_recs)
cf_seconds = time.perf_counter() - start

## Evaluate and save

In [3]:
hits = count_hits(data, recs)
summary = {
    "method": "popularity", "k": 1, "clusters": 1, "largest cluster %": 100.0, "avg neighbour pool": 0,
    **precision_recall(hits),
    "clustering s": 0.0,
    "CF s (eval users)": cf_seconds,
    "CF s (all users, estimated)": cf_seconds / len(data.eval_users) * data.n_users,
    "similarities (millions)": 0.0,
}
per_user = pd.DataFrame({"method": "popularity", "k": 1, "user": data.eval_users,
                         **{f"hits@{k}": hits[:, :k].sum(axis=1) for k in TOP_KS}})
results = save_results("popularity", [summary], [per_user])
results[["P@5", "P@10", "R@10", "P@20", "R@20", "CF s (all users, estimated)"]].round(4)

saved results\popularity.csv and results\popularity_per_user.csv


,P@5,P@10,R@10,P@20,R@20,"CF s (all users, estimated)"
0,0.0041,0.0041,0.0041,0.0037,0.0074,582.6637


In [4]:
names = pd.read_csv("cache/tracks.csv").iloc[:, 0]
most_popular = np.argsort(-data.popularity)[:10]
pd.DataFrame({"track": names[most_popular].to_numpy(), "users": data.popularity[most_popular].astype(int)},
             index=range(1, 11))

,track,users
1,Jigsaw Falling Into Place — Radiohead,9285
2,Pink + White — Frank Ocean,8966
3,No Surprises — Radiohead,8412
4,All I Need — Radiohead,7968
5,Nights — Frank Ocean,7442
6,Do I Wanna Know? — Arctic Monkeys,7375
7,"Good Luck, Babe! — Chappell Roan",7247
8,When the Sun Hits — Slowdive,7070
9,Cruel Summer — Taylor Swift,6892
10,Nude — Radiohead,6811
